<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
print(dataset)
print(len(dataset["train"]))

In [ ]:
import time

start = time.time()

for i, row in enumerate(dataset["train"]):
    if i % 100000 == 0:
        print(i, "rows processed")

    if i == 500000:
        break

print("Time:", time.time() - start)

In [ ]:
import unicodedata
import re

MAX_DOCS = 500000

with open("bn_news.txt", "w", encoding="utf-8") as f:

    for i, row in enumerate(dataset["train"]):

        text = unicodedata.normalize("NFC", row["text"])
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            f.write(text + "\n")

        if i + 1 >= MAX_DOCS:
            break

print("Saved", MAX_DOCS, "documents")

In [ ]:
word_count = 0
char_count = 0

with open("bn_news.txt", "r", encoding="utf-8") as f:
    for line in f:
        word_count += len(line.split())
        char_count += len(line)

print("Words:", word_count)
print("Characters:", char_count)
print("Approx MB:", char_count / 1024 / 1024)

In [ ]:
import gc

try:
    del dataset
except:
    pass

gc.collect()

In [ ]:
!pip -q install aksharamukha

In [ ]:
!ls -lh

In [ ]:
count = 0

with open("bn_news.txt", "r", encoding="utf-8") as f:
    for _ in f:
        count += 1

print("Documents:", count)

In [ ]:
import random

random.seed(42)

train_count = 0
test_count = 0

with open("bn_news.txt", "r", encoding="utf-8") as inp, \
     open("train_bn.txt", "w", encoding="utf-8") as train_f, \
     open("test_bn.txt", "w", encoding="utf-8") as test_f:

    for line in inp:

        line = line.strip()

        if not line:
            continue

        if random.random() < 0.1:
            test_f.write(line + "\n")
            test_count += 1
        else:
            train_f.write(line + "\n")
            train_count += 1

print("Train:", train_count)
print("Test:", test_count)

In [ ]:
import os

print("Train MB:", round(os.path.getsize("train_bn.txt")/1024/1024, 2))
print("Test MB :", round(os.path.getsize("test_bn.txt")/1024/1024, 2))

Transliteration

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()

count = 0

with open("train_bn.txt", "r", encoding="utf-8") as fin:
    for line in fin:
        line = line.strip()

        if not line:
            continue

        _ = transliterate.process(
            "Bengali",
            "ISO",
            line
        )

        count += 1

        if count == 1000:
            break

elapsed = time.time() - start

print("1000 docs:", elapsed, "seconds")
print("Docs/sec:", count / elapsed)

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()
count = 0

with open("train_bn.txt", "r", encoding="utf-8") as fin, \
     open("train_iso.txt", "w", encoding="utf-8") as fout:

    for line in fin:

        line = line.strip()

        if not line:
            continue

        iso = transliterate.process(
            "Bengali",
            "ISO",
            line
        )

        fout.write(iso + "\n")

        count += 1

        if count % 10000 == 0:
            elapsed = time.time() - start
            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print("Completed:", count)

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()
count = 0

with open("test_bn.txt", "r", encoding="utf-8") as fin, \
     open("test_iso.txt", "w", encoding="utf-8") as fout:

    for line in fin:

        line = line.strip()

        if not line:
            continue

        iso = transliterate.process(
            "Bengali",
            "ISO",
            line
        )

        fout.write(iso + "\n")

        count += 1

        if count % 10000 == 0:
            elapsed = time.time() - start
            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print("Completed:", count)

In [ ]:
for fname in [
    "train_bn.txt",
    "train_iso.txt"
]:
    print("\n---", fname, "---")

    with open(fname, "r", encoding="utf-8") as f:
        for _ in range(3):
            print(next(f).strip())

In [ ]:
import os

print("train_bn.txt :", round(os.path.getsize("train_bn.txt")/1024/1024,2), "MB")
print("train_iso.txt:", round(os.path.getsize("train_iso.txt")/1024/1024,2), "MB")

print("test_bn.txt :", round(os.path.getsize("test_bn.txt")/1024/1024,2), "MB")
print("test_iso.txt:", round(os.path.getsize("test_iso.txt")/1024/1024,2), "MB")

In [ ]:
from collections import Counter

def corpus_stats(filepath):
    total_chars = 0
    total_words = 0

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            words = line.split()

            total_chars += len(line)
            total_words += len(words)

    avg_word_len = total_chars / total_words

    return {
        "chars": total_chars,
        "words": total_words,
        "avg_word_len": avg_word_len
    }

bn_stats = corpus_stats("train_bn.txt")
iso_stats = corpus_stats("train_iso.txt")

print("Bengali:", bn_stats)
print("ISO:", iso_stats)

In [ ]:
expansion_factor = (
    iso_stats["chars"] /
    bn_stats["chars"]
)

print("Expansion Factor:", expansion_factor)

In [ ]:
def unique_vocab(filepath):

    vocab = set()

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:

            words = line.split()

            for w in words:
                vocab.add(w)

    return len(vocab)

bn_vocab = unique_vocab("train_bn.txt")
iso_vocab = unique_vocab("train_iso.txt")

print("BN Vocabulary:", bn_vocab)
print("ISO Vocabulary:", iso_vocab)

In [ ]:
vocab_ratio = iso_vocab / bn_vocab

print("Vocabulary Ratio:", vocab_ratio)

In [ ]:
import re

bengali_pattern = re.compile(r'[\u0980-\u09FF]')

total_lines = 0
converted_lines = 0

with open("train_iso.txt", "r", encoding="utf-8") as f:

    for line in f:

        total_lines += 1

        if not bengali_pattern.search(line):
            converted_lines += 1

coverage = converted_lines / total_lines

print("Coverage:", coverage)

In [ ]:
char_ratio = (
    iso_stats["chars"] /
    bn_stats["chars"]
)

print("Character Ratio:", char_ratio)

In [ ]:
print("\n=== TRANSLITERATION ANALYSIS ===\n")

print("BN Characters:", bn_stats["chars"])
print("ISO Characters:", iso_stats["chars"])

print()

print("BN Words:", bn_stats["words"])
print("ISO Words:", iso_stats["words"])

print()

print("BN Avg Word Length:",
      round(bn_stats["avg_word_len"], 3))

print("ISO Avg Word Length:",
      round(iso_stats["avg_word_len"], 3))

print()

print("BN Vocabulary:",
      bn_vocab)

print("ISO Vocabulary:",
      iso_vocab)

print()

print("Vocabulary Ratio:",
      round(vocab_ratio, 4))

print("Expansion Factor:",
      round(expansion_factor, 4))

print("Coverage:",
      round(coverage, 4))

In [ ]:
import re

bengali_pattern = re.compile(r'[\u0980-\u09FF]')

count = 0

with open("train_iso.txt", encoding="utf-8") as f:

    for line in f:

        if bengali_pattern.search(line):

            print(line[:300])

            count += 1

            if count == 20:
                break

In [ ]:
import re

# Bengali letters only
letter_pattern = re.compile(r'[অ-হািীুূৃেৈোৌংঃঁ]')

total_lines = 0
clean_lines = 0

with open("train_iso.txt", encoding="utf-8") as f:

    for line in f:

        total_lines += 1

        if not letter_pattern.search(line):
            clean_lines += 1

coverage = clean_lines / total_lines

print("Letter Coverage:", coverage)

In [ ]:
import re
from collections import Counter

pattern = re.compile(r'[\u0980-\u09FF]')

counter = Counter()

with open("train_iso.txt", encoding="utf-8") as f:

    for line in f:
        counter.update(pattern.findall(line))

print(counter.most_common(20))

In [ ]:
bn_ttr = bn_vocab / bn_stats["words"]
iso_ttr = iso_vocab / iso_stats["words"]

print("BN TTR :", round(bn_ttr, 6))
print("ISO TTR:", round(iso_ttr, 6))

Tokenization

In [ ]:
!pip -q install tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(
    BPE(
        unk_token="[UNK]"
    )
)

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=1,
    special_tokens=["[UNK]"]
)

tokenizer.train(
    ["train_bn.txt"],
    trainer
)

tokenizer.save("bpe_bn_5000.json")

print("Vocabulary Size:",
      tokenizer.get_vocab_size())

In [ ]:
sample = "নয়াদিল্লি: আজ ভ্যালেন্টাইনস ডে।"

enc = tokenizer.encode(sample)

print(enc.tokens)

In [ ]:
import numpy as np

def evaluate_tokenizer(tokenizer, filepath):

    total_words = 0
    total_subwords = 0
    total_chars = 0

    fragmented_words = 0
    unk_tokens = 0

    doc_token_counts = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            words = line.split()

            doc_tokens = 0

            for word in words:

                total_words += 1
                total_chars += len(word)

                enc = tokenizer.encode(word)

                n_subwords = len(enc.tokens)

                total_subwords += n_subwords
                doc_tokens += n_subwords

                if n_subwords > 1:
                    fragmented_words += 1

                unk_tokens += enc.tokens.count("[UNK]")

            doc_token_counts.append(doc_tokens)

    fertility = total_subwords / total_words

    cpt = total_chars / total_subwords

    compression = total_chars / total_subwords

    oov = unk_tokens / total_subwords

    avg_tokens = np.mean(doc_token_counts)

    wfr = fragmented_words / total_words

    variance = np.var(doc_token_counts)

    return {
        "vocab": tokenizer.get_vocab_size(),
        "oov": oov,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
print(tokenizer.token_to_id("[UNK]"))

In [ ]:
results = evaluate_tokenizer(
    tokenizer,
    "test_bn.txt"
)

print(results)

In [ ]:
from tokenizers.pre_tokenizers import ByteLevel

In [ ]:
samples = [
    "নয়াদিল্লি",
    "বাংলাদেশ",
    "কলকাতা",
    "ভ্যালেন্টাইনস",
    "ক্রিকেটার"
]

for s in samples:
    print(s)
    print(tokenizer.encode(s).tokens)
    print()

In [ ]:
word = "য়"

print(tokenizer.encode(word).tokens)

In [ ]:
print(tokenizer.token_to_id("য়"))

In [ ]:
from tokenizers.trainers import BpeTrainer

bengali_chars = list(
    "অআইঈউঊঋএঐওঔ"
    "কখগঘঙচছজঝঞ"
    "টঠডঢণতথদধন"
    "পফবভমযরলশষসহ"
    "ড়ঢ়য়ৎ"
    "ািীুূৃেৈোৌ"
    "ংঃঁ"
    "০১২৩৪৫৬৭৮৯"
    "।"
)

In [ ]:
tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")
)

tokenizer.pre_tokenizer = Whitespace()

tokenizer.train(
    ["train_bn.txt"],
    trainer
)

In [ ]:
trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=1,
    special_tokens=["[UNK]"],
    initial_alphabet=bengali_chars
)

In [ ]:
for ch in ["য়", "ড়", "ঢ়", "ৎ"]:
    print(ch, tokenizer.encode(ch).tokens)

In [ ]:
print(tokenizer.get_vocab_size())

In [ ]:
print(tokenizer.token_to_id("য়"))

In [ ]:
print(tokenizer.encode("নয়াদিল্লি").tokens)

In [ ]:
results = evaluate_tokenizer(
    tokenizer,
    "test_bn.txt"
)

print(results)

In [ ]:
import gc
import csv
import numpy as np

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
chars = set()

with open("train_bn.txt", encoding="utf-8") as f:
    for line in f:
        chars.update(line)

chars.discard(" ")
chars.discard("\n")

bengali_chars = sorted(list(chars))

print("Character inventory:", len(bengali_chars))

In [ ]:
def evaluate_tokenizer(tokenizer, filepath):

    total_words = 0
    total_subwords = 0
    total_chars = 0

    fragmented_words = 0
    unk_tokens = 0

    doc_token_counts = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            words = line.split()

            doc_tokens = 0

            for word in words:

                total_words += 1
                total_chars += len(word)

                enc = tokenizer.encode(word)

                n_subwords = len(enc.tokens)

                total_subwords += n_subwords
                doc_tokens += n_subwords

                if n_subwords > 1:
                    fragmented_words += 1

                unk_tokens += enc.tokens.count("[UNK]")

            doc_token_counts.append(doc_tokens)

    fertility = total_subwords / total_words

    cpt = total_chars / total_subwords

    compression = total_chars / total_subwords

    oov = unk_tokens / total_subwords

    avg_tokens = np.mean(doc_token_counts)

    wfr = fragmented_words / total_words

    variance = np.var(doc_token_counts)

    return {
        "vocab": tokenizer.get_vocab_size(),
        "oov": oov,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "bpe_bn_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=bengali_chars
    )

    tokenizer.train(
        ["train_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_bn_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_bn.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "BPE",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
with open("bpe_bn_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
chars = set()

with open("train_iso.txt", encoding="utf-8") as f:
    for line in f:
        chars.update(line)

chars.discard(" ")
chars.discard("\n")

iso_chars = sorted(list(chars))

print("Character inventory:", len(iso_chars))
print(iso_chars[:50])

In [ ]:
import gc
import csv

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "bpe_iso_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining ISO BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=iso_chars
    )

    tokenizer.train(
        ["train_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_iso_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_iso.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "ISO",
            "BPE",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
with open("bpe_iso_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import csv

bn = []
iso = []

with open("bpe_bn_results.csv") as f:
    bn = list(csv.DictReader(f))

with open("bpe_iso_results.csv") as f:
    iso = list(csv.DictReader(f))

print(
    f"{'Vocab':<8}"
    f"{'BN_Fert':<12}"
    f"{'ISO_Fert':<12}"
    f"{'BN_CPT':<12}"
    f"{'ISO_CPT':<12}"
)

for b, i in zip(bn, iso):

    print(
        f"{b['vocab']:<8}"
        f"{float(b['fertility']):<12.4f}"
        f"{float(i['fertility']):<12.4f}"
        f"{float(b['cpt']):<12.4f}"
        f"{float(i['cpt']):<12.4f}"
    )

In [ ]:
import gc
import csv

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "wp_bn_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining Bengali WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=bengali_chars
    )

    tokenizer.train(
        ["train_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_bn_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_bn.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "WordPiece",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
with open("wp_bn_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import gc
import csv

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "wp_iso_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining ISO WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=iso_chars
    )

    tokenizer.train(
        ["train_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_iso_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_iso.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "ISO",
            "WordPiece",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
!pip -q install sentencepiece

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_bn.txt",
        model_prefix=f"uni_bn_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")

In [ ]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(model_path, filepath):

    sp = spm.SentencePieceProcessor()
    sp.load(model_path)

    total_words = 0
    total_subwords = 0
    total_chars = 0

    fragmented_words = 0
    unk_tokens = 0

    doc_token_counts = []

    with open(filepath, encoding="utf-8") as f:

        for line in f:

            words = line.strip().split()

            doc_tokens = 0

            for word in words:

                total_words += 1
                total_chars += len(word)

                pieces = sp.encode(word, out_type=str)

                n_subwords = len(pieces)

                total_subwords += n_subwords
                doc_tokens += n_subwords

                if n_subwords > 1:
                    fragmented_words += 1

                unk_tokens += pieces.count("<unk>")

            doc_token_counts.append(doc_tokens)

    return {
        "vocab": sp.get_piece_size(),
        "oov": unk_tokens / total_subwords,
        "fertility": total_subwords / total_words,
        "cpt": total_chars / total_subwords,
        "compression": total_chars / total_subwords,
        "avg_tokens": np.mean(doc_token_counts),
        "wfr": fragmented_words / total_words,
        "variance": np.var(doc_token_counts)
    }

In [ ]:
import csv

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

with open("uni_bn_results.csv", "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    metrics = evaluate_unigram(
        f"uni_bn_{vocab_size}.model",
        "test_bn.txt"
    )

    print(metrics)

    with open("uni_bn_results.csv", "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "Unigram",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

print("Finished")

In [ ]:
with open("uni_bn_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_iso.txt",
        model_prefix=f"uni_iso_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")

In [ ]:
import csv

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

with open("uni_iso_results.csv", "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    metrics = evaluate_unigram(
        f"uni_iso_{vocab_size}.model",
        "test_iso.txt"
    )

    print(metrics)

    with open("uni_iso_results.csv", "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "Unigram",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

print("Finished")